# 05 — Detailed EDA on Cleaned Data (Probability & Distributions)

A closer look using ideas from the course: the empirical rule, distribution fits, the central limit theorem, bootstrapping, and a simple Bayesian estimate.

This notebook uses the cleaned dataset produced by `01_cleaning.ipynb` / `02_cleaning.ipynb`: `data/processed/cleaned.csv`. Run the cleaning notebook first.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.config import TARGET, FIGURES_DIR, TABLES_DIR, PROCESSED_DIR

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)


In [ ]:
# Load the cleaned dataset created by the cleaning notebook.
# This keeps detailed EDA consistent with statistical testing and modeling.
cleaned_path = PROCESSED_DIR / "cleaned.csv"

if not cleaned_path.exists():
    raise FileNotFoundError(
        f"{cleaned_path} was not found. Run the cleaning notebook first to create cleaned.csv."
    )

df = pd.read_csv(cleaned_path)

# Drop missing values only for the specific variables needed in each analysis.
bnp = df["bnp"].dropna().values
sodium = df["sodium"].dropna().values
age = df["age"].dropna().values

df.shape


In [ ]:
# empirical rule on age vs what a normal distribution predicts
m, s = age.mean(), age.std(ddof=1)
rows = []
for k in (1, 2, 3):
    inside = np.mean((age >= m - k * s) & (age <= m + k * s)) * 100
    normal = (stats.norm.cdf(k) - stats.norm.cdf(-k)) * 100
    rows.append({'k_sd': k, 'empirical_pct': round(inside, 2), 'normal_pct': round(normal, 2)})
emp = pd.DataFrame(rows)
emp.to_csv(TABLES_DIR / 'tbl_empirical_rule_age.csv', index=False)
emp

In [ ]:
mu, sd = stats.norm.fit(sodium)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(sodium, bins=30, density=True, alpha=0.6, color='steelblue')
xs = np.linspace(sodium.min(), sodium.max(), 200)
ax.plot(xs, stats.norm.pdf(xs, mu, sd), 'r-', lw=2)
ax.set_title(f'Sodium with normal fit (mu={mu:.1f}, sd={sd:.1f})')
ax.set_xlabel('sodium')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_dist_normal_sodium.png', dpi=120)

In [ ]:
a, loc, scale = stats.gamma.fit(bnp, floc=0)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(bnp, bins=40, density=True, alpha=0.6, color='seagreen')
xs = np.linspace(bnp.min(), bnp.max(), 200)
ax.plot(xs, stats.gamma.pdf(xs, a, loc=loc, scale=scale), 'r-', lw=2)
ax.set_title(f'BNP with gamma fit (shape={a:.2f})')
ax.set_xlabel('bnp')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_dist_gamma_bnp.png', dpi=120)

In [ ]:
# central limit theorem: sampling distribution of the mean BNP
means = np.array([rng.choice(bnp, size=50).mean() for _ in range(2000)])
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(means, bins=40, density=True, alpha=0.6, color='slateblue')
xs = np.linspace(means.min(), means.max(), 200)
ax.plot(xs, stats.norm.pdf(xs, means.mean(), means.std()), 'r-', lw=2)
ax.set_title('Sampling distribution of mean BNP (n=50)')
ax.set_xlabel('sample mean bnp')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_clt_bnp.png', dpi=120)

In [ ]:
# bootstrap 95% confidence interval for the mean BNP
boot = np.array([rng.choice(bnp, size=len(bnp)).mean() for _ in range(2000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(boot, bins=40, color='darkorange', alpha=0.7)
ax.axvline(lo, color='black', ls='--')
ax.axvline(hi, color='black', ls='--')
ax.set_title(f'Bootstrap mean BNP: 95% CI [{lo:.1f}, {hi:.1f}]')
ax.set_xlabel('bootstrap mean bnp')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_bootstrap_bnp.png', dpi=120)

In [ ]:
# Bayesian estimate of the readmission rate (Beta-Binomial, uniform prior)
k = int(df[TARGET].sum())
n = len(df)
post = stats.beta(1 + k, 1 + n - k)
lo, hi = post.ppf(0.025), post.ppf(0.975)
xs = np.linspace(0.35, 0.47, 300)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(xs, post.pdf(xs), 'b-', lw=2)
ax.axvline(lo, color='grey', ls='--')
ax.axvline(hi, color='grey', ls='--')
ax.set_title(f'Posterior readmission rate (mean={post.mean():.3f}, 95% [{lo:.3f}, {hi:.3f}])')
ax.set_xlabel('P(readmitted within 30 days)')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_bayes_proportion.png', dpi=120)

In [ ]:
missing_summary = df.isna().mean().mul(100).round(2)
missing_cols = missing_summary[missing_summary > 0].to_dict()

findings = pd.DataFrame([
    {"topic": "class balance", "finding": f"{k}/{n} ({k / n:.1%}) readmitted within 30 days"},
    {"topic": "missing data", "finding": f"Remaining missingness after cleaning: {missing_cols}"},
    {"topic": "bnp distribution", "finding": f"right-skewed, gamma shape ~{a:.2f}"},
    {"topic": "sodium distribution", "finding": "roughly normal after cleaning; used for the normal fit"},
    {"topic": "age empirical rule", "finding": f"~{emp.loc[0, 'empirical_pct']}% within 1 sd vs 68% for a normal"},
])
findings.to_csv(TABLES_DIR / "tbl_eda_detailed_findings_cleaned.csv", index=False)
findings
